[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/badrelkhalyly-etu-alt/ability-adjusted-mcq-profiling/blob/main/notebooks/04_lca_mixirt.ipynb)

# E7 and contingency tables: ability-adjusted classes, stability, cross-classification

**Objective.** Class-specific difficulty deviations after ability adjustment, bootstrap stability, and contingency tables of K-means profiles x classes x score quartile x OECD x booklet.

**Data.** The analysis-ready PISA 2025 public-use extracts are in `data/processed/` (see the Data section of `README.md`), so cloning the repository in Colab is enough to run the real analyses; `DEMO` switches to real data automatically. Set `DEMO = True` for a fast synthetic demonstration. **Demo outputs are not findings.** The *saved results* cells display the PISA 2025 aggregate outputs committed in `results/`.

Errors are raised, never swallowed. Every output is written to `results/` with the form name; outputs from synthetic data are prefixed `demo_`.

In [ ]:
# --- Environment (Colab or local) -------------------------------------------
REPO_URL = "https://github.com/badrelkhalyly-etu-alt/ability-adjusted-mcq-profiling.git"   # GitHub repository; a private repository needs a token in the URL or an uploaded copy
DEMO = None            # None = real PISA 2025 data if data/processed/ is present (it is in the repository), else synthetic; or force True/False
import os, sys, subprocess, platform
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("ability-adjusted-mcq-profiling"):
        if not REPO_URL:
            raise SystemExit("Set REPO_URL or upload the repository folder to /content.")
        subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL], check=True)
    os.chdir("ability-adjusted-mcq-profiling")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.exists("configs") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
if DEMO is None:
    DEMO = not os.path.exists("data/processed/math_formA.parquet")
import numpy, scipy, pandas, sklearn
print(platform.python_version(), "numpy", numpy.__version__, "scipy", scipy.__version__,
      "pandas", pandas.__version__, "sklearn", sklearn.__version__)
FORM = "demo" if DEMO else "math_formA"
print("Mode:", "SYNTHETIC DEMONSTRATION - not empirical results" if DEMO else "REAL PISA 2025 data")

In [ ]:
import subprocess, sys
def run(stage, form=None):
    cmd = [sys.executable, "scripts/run_experiments.py", "--stage", stage] + (["--demo"] if DEMO else []) \
          + (["--form", form] if form else [])
    r = subprocess.run(cmd, capture_output=True, text=True)
    print(r.stdout[-6000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"stage {stage} failed")

In [ ]:
run('stability')

In [ ]:
if not DEMO:
    run('contingency')

In [ ]:
# Saved PISA 2025 results (aggregate outputs committed to the repository; readable without the licensed data)
import json, pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown
def show_json(name, keys=None):
    p = Path("results/metrics") / name
    if not p.exists():
        print("missing", p); return
    d = json.loads(p.read_text(encoding="utf8"))
    print(json.dumps({k: d[k] for k in keys if k in d} if keys else d, indent=1, default=str)[:5000])
def show_csv(name):
    p = Path("results/tables") / name
    display(pd.read_csv(p)) if p.exists() else print("missing", p)
def show_fig(name):
    p = Path("results/figures") / f"{name}.png"
    display(Image(str(p))) if p.exists() else print("missing", p)

In [ ]:
show_json('math_formA_E7_stability.json', ['B', 'ari_median', 'ari_q', 'dev_corr_median', 'aris'])

In [ ]:
for t in ['profile_x_M1_3', 'M1_3_x_scoreband', 'profile_x_M1_2', 'profile_x_oecd', 'profile_x_booklet', 'M1_3_x_oecd']:
    display(Markdown(f'**{t}**: counts, then row %'))
    show_csv(f'math_formA_ct_{t}.csv'); show_csv(f'math_formA_ct_{t}_rowpct.csv')
c = json.loads(Path('results/metrics/math_formA_contingency.json').read_text())
display(pd.DataFrame({k: v['stats'] for k, v in c.items() if isinstance(v, dict) and 'stats' in v}).T)
show_csv('math_formA_M1_3_class_profile.csv')

In [ ]:
show_fig('fig07_10_class_profiles_stability')